# Creating the initial model

In [10]:
import pandas as pd

In [25]:
import pandas as pd
import numpy as np
from collections import defaultdict
from io import StringIO

seasons = ['2016-17', '2017-18', '2018-19', '2019-20', '2020-21', 
           '2021-22', '2022-23', '2023-24', '2024-25']

season_columns = {}  # season -> set of columns

# Load seasons except 2024-25 normally and store columns
for season in seasons:
    path = f'../data/{season}/gws/merged_gw.csv'
    try:
        if season == '2024-25':
            # Special 2024-25 handling below
            continue
        
        df = pd.read_csv(path, encoding="ISO-8859-1")
        season_columns[season] = set(df.columns)
        print(f"✅ Loaded {season} with {len(df.columns)} columns")
    except Exception as e:
        print(f"❌ Error loading {season}: {e}")

# Special handling for 2024-25 with broken rows and extra columns
path_2024 = '../data/2024-25/gws/merged_gw.csv'

with open(path_2024, encoding='ISO-8859-1') as f:
    lines = f.readlines()

header = lines[0].strip().split(',')
expected_cols = len(header)

extra_cols = ['mng_clean_sheets', 'mng_draw', 'mng_goals_scored', 'mng_loss',
              'mng_underdog_draw', 'mng_underdog_win', 'mng_win']

insert_after = 'minutes'
insert_idx = header.index(insert_after) + 1
full_header = header[:insert_idx] + extra_cols + header[insert_idx:]

# Use 0-based indexing on lines list:
# Header: lines[0]
# First part: lines[1:14179]  # up to line 14178 (14179 not included)
# Second part: lines[14179:]  # from line 14179 onward

first_part_str = ''.join(lines[1:14179])  # up to line 14178
first_part = pd.read_csv(StringIO(first_part_str), names=header)

# Insert extra columns at correct position with NaNs
for i, col in enumerate(extra_cols):
    first_part.insert(insert_idx + i, col, np.nan)

second_part_str = ''.join(lines[14179:])  # from line 14179 onward
second_part = pd.read_csv(StringIO(second_part_str), names=full_header)

df_2024 = pd.concat([first_part, second_part], ignore_index=True)

season_columns['2024-25'] = set(df_2024.columns)

print(f"✅ Loaded 2024-25 with {len(df_2024.columns)} columns after fix")

# Combine all columns across all seasons
all_columns = set().union(*season_columns.values())

# Determine consistent columns (present in all seasons)
if len(season_columns) == 1:
    consistent_columns = next(iter(season_columns.values()))
else:
    consistent_columns = set.intersection(*season_columns.values())

# Inconsistent columns (appear only in some seasons)
inconsistent_columns = all_columns - consistent_columns

# Map columns to seasons they appear in
column_season_map = defaultdict(list)
for season, cols in season_columns.items():
    for col in cols:
        column_season_map[col].append(season)

# Add suffixes for inconsistent columns
inconsistent_columns_with_seasons = []
for col in inconsistent_columns:
    for season in column_season_map[col]:
        inconsistent_columns_with_seasons.append(f"{col}_{season}")

print("\n✅ CONSISTENT COLUMNS:")
print(sorted(consistent_columns))

print("\n⚠️  INCONSISTENT COLUMNS with season suffix:")
print(sorted(inconsistent_columns_with_seasons))


✅ Loaded 2016-17 with 56 columns
✅ Loaded 2017-18 with 56 columns
✅ Loaded 2018-19 with 56 columns
✅ Loaded 2019-20 with 33 columns
✅ Loaded 2020-21 with 36 columns
✅ Loaded 2021-22 with 36 columns
✅ Loaded 2022-23 with 41 columns
✅ Loaded 2023-24 with 41 columns
✅ Loaded 2024-25 with 49 columns after fix

✅ CONSISTENT COLUMNS:
['GW', 'assists', 'bonus', 'bps', 'clean_sheets', 'creativity', 'element', 'fixture', 'goals_conceded', 'goals_scored', 'ict_index', 'influence', 'kickoff_time', 'minutes', 'name', 'opponent_team', 'own_goals', 'penalties_missed', 'penalties_saved', 'red_cards', 'round', 'saves', 'selected', 'team_a_score', 'team_h_score', 'threat', 'total_points', 'transfers_balance', 'transfers_in', 'transfers_out', 'value', 'was_home', 'yellow_cards']

⚠️  INCONSISTENT COLUMNS with season suffix:
['attempted_passes_2016-17', 'attempted_passes_2017-18', 'attempted_passes_2018-19', 'big_chances_created_2016-17', 'big_chances_created_2017-18', 'big_chances_created_2018-19', 'big

In [28]:
df_2024.columns.tolist()

['name',
 'position',
 'team',
 'xP',
 'assists',
 'bonus',
 'bps',
 'clean_sheets',
 'creativity',
 'element',
 'expected_assists',
 'expected_goal_involvements',
 'expected_goals',
 'expected_goals_conceded',
 'fixture',
 'goals_conceded',
 'goals_scored',
 'ict_index',
 'influence',
 'kickoff_time',
 'minutes',
 'mng_clean_sheets',
 'mng_draw',
 'mng_goals_scored',
 'mng_loss',
 'mng_underdog_draw',
 'mng_underdog_win',
 'mng_win',
 'modified',
 'opponent_team',
 'own_goals',
 'penalties_missed',
 'penalties_saved',
 'red_cards',
 'round',
 'saves',
 'selected',
 'starts',
 'team_a_score',
 'team_h_score',
 'threat',
 'total_points',
 'transfers_balance',
 'transfers_in',
 'transfers_out',
 'value',
 'was_home',
 'yellow_cards',
 'GW']

In [35]:
# Show all columns without truncation
pd.set_option('display.max_columns', None)

df_2024.head(50)

,name,position,team,xP,assists,bonus,bps,clean_sheets,creativity,element,expected_assists,expected_goal_involvements,expected_goals,expected_goals_conceded,fixture,goals_conceded,goals_scored,ict_index,influence,kickoff_time,minutes,mng_clean_sheets,mng_draw,mng_goals_scored,mng_loss,mng_underdog_draw,mng_underdog_win,mng_win,modified,opponent_team,own_goals,penalties_missed,penalties_saved,red_cards,round,saves,selected,starts,team_a_score,team_h_score,threat,total_points,transfers_balance,transfers_in,transfers_out,value,was_home,yellow_cards,GW
0,Alex Scott,MID,Bournemouth,1.6,0,0,11,0,12.8,77,0.01,0.01,0.00,1.02,6,1,0,3.6,22.8,2024-08-17T14:00:00Z,62,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,16,0,0,0,0,1,0,4339,1,1,1,0.0,2,0,0,0,50,False,0,1
1,Carlos Miguel dos Santos Pereira,GK,Nott'm Forest,2.2,0,0,0,0,0.0,427,0.00,0.00,0.00,0.00,6,0,0,0.0,0.0,2024-08-17T14:00:00Z,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,3,0,0,0,0,1,0,33324,0,1,1,0.0,0,0,0,0,45,True,0,1
2,Tomiyasu Takehiro,DEF,Arsenal,0.0,0,0,0,0,0.0,22,0.00,0.00,0.00,0.00,2,0,0,0.0,0.0,2024-08-17T14:00:00Z,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,20,0,0,0,0,1,0,8462,0,0,2,0.0,0,0,0,0,50,True,0,1
3,Malcolm Ebiowei,MID,Crystal Palace,0.0,0,0,0,0,0.0,197,0.00,0.00,0.00,0.00,8,0,0,0.0,0.0,2024-08-18T13:00:00Z,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,4,0,0,0,0,1,0,716,0,1,2,0.0,0,0,0,0,45,False,0,1
4,Ben Brereton DÃ­az,MID,Southampton,1.0,0,0,-2,0,14.0,584,0.02,0.32,0.30,0.25,5,1,0,3.3,2.6,2024-08-17T14:00:00Z,70,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,15,0,0,0,0,1,0,66244,1,0,1,16.0,1,0,0,0,55,False,1,1
5,Pau Torres,DEF,Aston Villa,1.9,0,0,17,0,1.9,52,0.01,0.01,0.00,2.46,7,1,0,3.1,29.2,2024-08-17T16:30:00Z,90,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,19,0,0,0,0,1,0,122710,1,2,1,0.0,2,0,0,0,45,False,0,1
6,Joel Ward,DEF,Crystal Palace,1.9,0,0,0,0,0.0,215,0.00,0.00,0.00,0.00,8,0,0,0.0,0.0,2024-08-18T13:00:00Z,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,4,0,0,0,0,1,0,12469,0,1,2,0.0,0,0,0,0,45,False,0,1
7,Will Lankshear,FWD,Spurs,1.5,0,0,0,0,0.0,609,0.00,0.00,0.00,0.00,10,0,0,0.0,0.0,2024-08-19T19:00:00Z,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,11,0,0,0,0,1,0,17747,0,1,1,0.0,0,0,0,0,45,False,0,1
8,Hwang Hee-chan,MID,Wolves,1.3,0,0,14,0,16.3,550,0.11,0.11,0.00,1.24,2,2,0,2.6,6.0,2024-08-17T14:00:00Z,90,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,1,0,0,0,0,1,0,83019,1,0,2,4.0,2,0,0,0,65,False,0,1
9,Mikey Moore,MID,Spurs,1.5,0,0,0,0,0.0,610,0.00,0.00,0.00,0.00,10,0,0,0.0,0.0,2024-08-19T19:00:00Z,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,11,0,0,0,0,1,0,8413,0,1,1,0.0,0,0,0,0,45,False,0,1


In [33]:
inconsistent_columns

{'attempted_passes',
 'big_chances_created',
 'big_chances_missed',
 'clearances_blocks_interceptions',
 'completed_passes',
 'dribbles',
 'ea_index',
 'errors_leading_to_goal',
 'errors_leading_to_goal_attempt',
 'expected_assists',
 'expected_goal_involvements',
 'expected_goals',
 'expected_goals_conceded',
 'fouls',
 'id',
 'key_passes',
 'kickoff_time_formatted',
 'loaned_in',
 'loaned_out',
 'mng_clean_sheets',
 'mng_draw',
 'mng_goals_scored',
 'mng_loss',
 'mng_underdog_draw',
 'mng_underdog_win',
 'mng_win',
 'modified',
 'offside',
 'open_play_crosses',
 'penalties_conceded',
 'position',
 'recoveries',
 'starts',
 'tackled',
 'tackles',
 'target_missed',
 'team',
 'winning_goals',
 'xP'}

First let's decipher the different columns of data provided with each season

In [15]:
seasons = ['2016-17', '2017-18', '2018-19', '2019-20', '2020-21', 
           '2021-22', '2022-23', '2023-24']

for season in seasons:
    path = f'../data/{season}/gws/merged_gw.csv'
    
    try:
        if season == '2024-25':
            # Handle special case: 49 columns and malformed rows
            df = pd.read_csv(path, encoding="ISO-8859-1", on_bad_lines='skip')
        else:
            df = pd.read_csv(path, encoding="ISO-8859-1")

        print(f"\n✅ Season {season} loaded successfully. Columns: {len(df.columns)}")
        print(df.columns.tolist())

    except Exception as e:
        print(f"\n❌ Failed to load {season}: {e}")




✅ Season 2016-17 loaded successfully. Columns: 56
['name', 'assists', 'attempted_passes', 'big_chances_created', 'big_chances_missed', 'bonus', 'bps', 'clean_sheets', 'clearances_blocks_interceptions', 'completed_passes', 'creativity', 'dribbles', 'ea_index', 'element', 'errors_leading_to_goal', 'errors_leading_to_goal_attempt', 'fixture', 'fouls', 'goals_conceded', 'goals_scored', 'ict_index', 'id', 'influence', 'key_passes', 'kickoff_time', 'kickoff_time_formatted', 'loaned_in', 'loaned_out', 'minutes', 'offside', 'open_play_crosses', 'opponent_team', 'own_goals', 'penalties_conceded', 'penalties_missed', 'penalties_saved', 'recoveries', 'red_cards', 'round', 'saves', 'selected', 'tackled', 'tackles', 'target_missed', 'team_a_score', 'team_h_score', 'threat', 'total_points', 'transfers_balance', 'transfers_in', 'transfers_out', 'value', 'was_home', 'winning_goals', 'yellow_cards', 'GW']

✅ Season 2017-18 loaded successfully. Columns: 56
['name', 'assists', 'attempted_passes', 'big_c

In [16]:
import pandas as pd
from collections import defaultdict

seasons = ['2016-17', '2017-18', '2018-19', '2019-20', '2020-21', 
           '2021-22', '2022-23', '2023-24', '2024-25']

season_columns = {}  # season -> set of columns

# Load each season and store its column set
for season in seasons:
    path = f'../data/{season}/gws/merged_gw.csv'
    try:
        df = pd.read_csv(path, encoding="ISO-8859-1")
        season_columns[season] = set(df.columns)
        print(f"✅ Loaded {season} with {len(df.columns)} columns")
    except Exception as e:
        print(f"❌ Error loading {season}: {e}")

# Combine all columns across all seasons
all_columns = set().union(*season_columns.values())

# Determine which columns are consistent (appear in every season)
consistent_columns = set.intersection(*season_columns.values())

# Identify inconsistent columns
inconsistent_columns = all_columns - consistent_columns

# Build a dict: column -> [seasons it appears in]
column_season_map = defaultdict(list)
for season, cols in season_columns.items():
    for col in cols:
        column_season_map[col].append(season)

# Build list with suffixes for inconsistent columns
inconsistent_columns_with_seasons = []
for col in inconsistent_columns:
    for season in column_season_map[col]:
        inconsistent_columns_with_seasons.append(f"{col}_{season}")

# Show results
print("\n✅ CONSISTENT COLUMNS:")
print(sorted(consistent_columns))

print("\n⚠️  INCONSISTENT COLUMNS with season suffix:")
print(sorted(inconsistent_columns_with_seasons))



✅ Loaded 2016-17 with 56 columns
✅ Loaded 2017-18 with 56 columns
✅ Loaded 2018-19 with 56 columns
✅ Loaded 2019-20 with 33 columns
✅ Loaded 2020-21 with 36 columns
✅ Loaded 2021-22 with 36 columns
✅ Loaded 2022-23 with 41 columns
✅ Loaded 2023-24 with 41 columns
❌ Error loading 2024-25: Error tokenizing data. C error: Expected 42 fields in line 14180, saw 49


✅ CONSISTENT COLUMNS:
['GW', 'assists', 'bonus', 'bps', 'clean_sheets', 'creativity', 'element', 'fixture', 'goals_conceded', 'goals_scored', 'ict_index', 'influence', 'kickoff_time', 'minutes', 'name', 'opponent_team', 'own_goals', 'penalties_missed', 'penalties_saved', 'red_cards', 'round', 'saves', 'selected', 'team_a_score', 'team_h_score', 'threat', 'total_points', 'transfers_balance', 'transfers_in', 'transfers_out', 'value', 'was_home', 'yellow_cards']

⚠️  INCONSISTENT COLUMNS with season suffix:
['attempted_passes_2016-17', 'attempted_passes_2017-18', 'attempted_passes_2018-19', 'big_chances_created_2016-17', 'big_chanc

In [21]:
import pandas as pd
import numpy as np
from io import StringIO

path = '../data/2024-25/gws/merged_gw.csv'

# Read all lines
with open(path, encoding='ISO-8859-1') as f:
    lines = f.readlines()

# Split header
header = lines[0].strip().split(',')
expected_cols = len(header)

# Define extra columns and final column order
extra_cols = ['mng_clean_sheets', 'mng_draw', 'mng_goals_scored', 'mng_loss',
              'mng_underdog_draw', 'mng_underdog_win', 'mng_win']

# Find insert position — we know 'minutes' comes before them
insert_after = 'minutes'
insert_idx = header.index(insert_after) + 1

# Final consistent header (42 + 7 = 49)
full_header = header[:insert_idx] + extra_cols + header[insert_idx:]

# FIRST PART
first_part_str = ''.join(lines[1:14179])
first_part = pd.read_csv(StringIO(first_part_str), names=header)

# Insert mng_* columns in the right position
for col in extra_cols:
    first_part.insert(insert_idx, col, np.nan)
    insert_idx += 1

# SECOND PART
second_part_str = ''.join(lines[14179:])
second_part = pd.read_csv(StringIO(second_part_str), names=full_header)

# COMBINE BOTH
df_2024 = pd.concat([first_part, second_part], ignore_index=True)

print(f"✅ Final 2024–25 shape: {df_2024.shape}")
print(f"📦 Final columns (ordered):\n{df_2024.columns.tolist()}")

✅ Final 2024–25 shape: (27605, 49)
📦 Final columns (ordered):
['name', 'position', 'team', 'xP', 'assists', 'bonus', 'bps', 'clean_sheets', 'creativity', 'element', 'expected_assists', 'expected_goal_involvements', 'expected_goals', 'expected_goals_conceded', 'fixture', 'goals_conceded', 'goals_scored', 'ict_index', 'influence', 'kickoff_time', 'minutes', 'mng_clean_sheets', 'mng_draw', 'mng_goals_scored', 'mng_loss', 'mng_underdog_draw', 'mng_underdog_win', 'mng_win', 'modified', 'opponent_team', 'own_goals', 'penalties_missed', 'penalties_saved', 'red_cards', 'round', 'saves', 'selected', 'starts', 'team_a_score', 'team_h_score', 'threat', 'total_points', 'transfers_balance', 'transfers_in', 'transfers_out', 'value', 'was_home', 'yellow_cards', 'GW']


In [22]:
import pandas as pd
import numpy as np
from io import StringIO

# Define paths & seasons
seasons = [
    '2016-17', '2017-18', '2018-19', '2019-20', '2020-21', 
    '2021-22', '2022-23', '2023-24', '2024-25'
]

paths = {
    '2016-17': 'path/to/2016-17.csv',
    '2017-18': 'path/to/2017-18.csv',
    '2018-19': 'path/to/2018-19.csv',
    '2019-20': 'path/to/2019-20.csv',
    '2020-21': 'path/to/2020-21.csv',
    '2021-22': 'path/to/2021-22.csv',
    '2022-23': 'path/to/2022-23.csv',
    '2023-24': 'path/to/2023-24.csv',
    '2024-25': 'path/to/2024-25/gws/merged_gw.csv'
}

def read_season_df(season):
    if season != '2024-25':
        df = pd.read_csv(paths[season], encoding='ISO-8859-1')
        return df
    else:
        # 2024-25 fix (split, add mng columns in right place, concat)
        with open(paths[season], encoding='ISO-8859-1') as f:
            lines = f.readlines()

        header = lines[0].strip().split(',')
        extra_cols = ['mng_clean_sheets', 'mng_draw', 'mng_goals_scored', 'mng_loss',
                      'mng_underdog_draw', 'mng_underdog_win', 'mng_win']
        insert_after = 'minutes'
        insert_idx = header.index(insert_after) + 1
        full_header = header[:insert_idx] + extra_cols + header[insert_idx:]

        # Read first part
        first_part_str = ''.join(lines[1:14179])
        first_part = pd.read_csv(StringIO(first_part_str), names=header)
        for col in extra_cols:
            first_part.insert(insert_idx, col, np.nan)
            insert_idx += 1

        # Read second part
        second_part_str = ''.join(lines[14179:])
        second_part = pd.read_csv(StringIO(second_part_str), names=full_header)

        # Concatenate
        df = pd.concat([first_part, second_part], ignore_index=True)
        return df

# Read all seasons
season_columns = {}
for season in seasons:
    try:
        df = read_season_df(season)
        cols = set(df.columns)
        season_columns[season] = cols
        print(f"✅ Loaded {season} with {len(cols)} columns")
    except Exception as e:
        print(f"❌ Error loading {season}: {e}")

# Find consistent columns (present in all seasons)
consistent_cols = set.intersection(*season_columns.values())

# Find inconsistent columns with season suffixes
inconsistent_cols = []
for season, cols in season_columns.items():
    for col in cols:
        if col not in consistent_cols:
            inconsistent_cols.append(f"{col}_{season}")

print(f"\n✅ CONSISTENT COLUMNS:\n{sorted(consistent_cols)}\n")
print(f"⚠️ INCONSISTENT COLUMNS with season suffix:\n{sorted(inconsistent_cols)}")


❌ Error loading 2016-17: [Errno 2] No such file or directory: 'path/to/2016-17.csv'
❌ Error loading 2017-18: [Errno 2] No such file or directory: 'path/to/2017-18.csv'
❌ Error loading 2018-19: [Errno 2] No such file or directory: 'path/to/2018-19.csv'
❌ Error loading 2019-20: [Errno 2] No such file or directory: 'path/to/2019-20.csv'
❌ Error loading 2020-21: [Errno 2] No such file or directory: 'path/to/2020-21.csv'
❌ Error loading 2021-22: [Errno 2] No such file or directory: 'path/to/2021-22.csv'
❌ Error loading 2022-23: [Errno 2] No such file or directory: 'path/to/2022-23.csv'
❌ Error loading 2023-24: [Errno 2] No such file or directory: 'path/to/2023-24.csv'
❌ Error loading 2024-25: [Errno 2] No such file or directory: 'path/to/2024-25/gws/merged_gw.csv'


TypeError: unbound method set.intersection() needs an argument

In [23]:
print("Seasons loaded:", list(season_columns.keys()))
print("Number of column sets:", len(season_columns.values()))


Seasons loaded: []
Number of column sets: 0


In [ ]:
# Combine season data

seasons = ['2016-17', '2017-18','2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24', '2024-25']
dfs = []

for season in seasons:
    path = f'../data/{season}/gws/merged_gw.csv'
    df = pd.read_csv(path, encoding="ISO-8859-1")
    df['season'] = season
    dfs.append(df)

all_data = pd.concat(dfs)
all_data.reset_index(drop=True, inplace=True)

In [ ]:
idea:

# FPL Model Training Pipeline (Option 1: Holdout Season)

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import joblib

# ------------------------------
# STEP 1: Load and Merge All Seasons
# ------------------------------
seasons = [
    '2016-17', '2017-18', '2018-19',
    '2019-20', '2020-21', '2021-22',
    '2022-23', '2023-24'
]

dfs = []
for season in seasons:
    path = f'../data/{season}/gws/merged_gw.csv'
    try:
        df = pd.read_csv(path, encoding='ISO-8859-1')
        df['season'] = season
        dfs.append(df)
    except Exception as e:
        print(f"Could not load {season}: {e}")

df_all = pd.concat(dfs, ignore_index=True)

# ------------------------------
# STEP 2: Clean Columns + Feature Engineering
# ------------------------------
# Only keep rows where player played

df_all = df_all[df_all['minutes'] > 0].copy()
df_all = df_all.sort_values(['name', 'season', 'round'])

# Lag Features (simulate knowledge before next GW)
df_all['points_last_1'] = df_all.groupby('name')['total_points'].shift(1)
df_all['points_last_3_avg'] = df_all.groupby('name')['total_points'].shift(1).rolling(3).mean().reset_index(level=0, drop=True)
df_all['minutes_last_1'] = df_all.groupby('name')['minutes'].shift(1)
df_all['goals_last_3'] = df_all.groupby('name')['goals_scored'].shift(1).rolling(3).sum().reset_index(level=0, drop=True)

# Target: next GW points
df_all['target'] = df_all.groupby('name')['total_points'].shift(-1)

# Drop rows with missing lag or target
df_all = df_all.dropna(subset=['points_last_1', 'points_last_3_avg', 'minutes_last_1', 'goals_last_3', 'target'])

# ------------------------------
# STEP 3: Add Season Weights
# ------------------------------
season_weights = {
    '2016-17': 0.2,
    '2017-18': 0.4,
    '2018-19': 0.6,
    '2019-20': 0.8,
    '2020-21': 1.0,
    '2021-22': 1.2,
    '2022-23': 1.5,
    '2023-24': 2.0
}
df_all['season_weight'] = df_all['season'].map(season_weights)

# ------------------------------
# STEP 4: Train/Test Split (Holdout 2023-24)
# ------------------------------
train_df = df_all[df_all['season'] != '2023-24']
test_df = df_all[df_all['season'] == '2023-24']

features = [
    'points_last_1', 'points_last_3_avg',
    'minutes_last_1', 'goals_last_3',
    'now_cost', 'was_home', 'selected_by_percent'
]

X_train = train_df[features]
y_train = train_df['target']
weights = train_df['season_weight']

X_test = test_df[features]
y_test = test_df['target']

# ------------------------------
# STEP 5: Train Model
# ------------------------------
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train, sample_weight=weights)

# Save the model
joblib.dump(model, 'fpl_base_model.pkl')

# ------------------------------
# STEP 6: Evaluate
# ------------------------------
y_pred = model.predict(X_test)
print("\n--- Evaluation on 2023-24 Season ---")
print("RMSE:", mean_squared_error(y_test, y_pred, squared=False))
print("R2:", r2_score(y_test, y_pred))

# View top predictions
test_df = test_df.copy()
test_df['predicted'] = y_pred
print(test_df[['name', 'round', 'total_points', 'predicted']].sort_values('predicted', ascending=False).head(10))


SyntaxError: invalid syntax (425084607.py, line 1)

In [1]:
import pandas as pd

In [7]:
# Combine season data

seasons = ['2018-19', '2019-20', '2020-21', '2021-22', '2022-23']
dfs = []

for season in seasons:
    path = f'../data/{season}/gws/merged_gw.csv'
    df = pd.read_csv(path, encoding="ISO-8859-1")
    df['season'] = season
    dfs.append(df)

all_data = pd.concat(dfs)
all_data.reset_index(drop=True, inplace=True)

In [8]:

all_data

,name,assists,attempted_passes,big_chances_created,big_chances_missed,bonus,bps,clean_sheets,clearances_blocks_interceptions,completed_passes,...,GW,season,position,team,xP,expected_assists,expected_goal_involvements,expected_goals,expected_goals_conceded,starts
0,Aaron_Cresswell_402,0,0.0,0.0,0.0,0,0,0,0.0,0.0,...,1,2018-19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Aaron_Lennon_83,0,22.0,0.0,1.0,0,6,1,1.0,17.0,...,1,2018-19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Aaron_Mooy_199,0,51.0,0.0,0.0,0,24,0,2.0,40.0,...,1,2018-19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Aaron_Ramsey_14,0,11.0,0.0,0.0,0,7,0,0.0,7.0,...,1,2018-19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Aaron_Wan-Bissaka_145,1,29.0,1.0,0.0,3,38,1,11.0,19.0,...,1,2018-19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
120662,Oliver Skipp,0,NaN,NaN,NaN,0,16,0,NaN,NaN,...,38,2022-23,MID,Spurs,2.0,0.01,0.01,0.0,1.5,1.0
120663,Ryan Sessegnon,0,NaN,NaN,NaN,0,0,0,NaN,NaN,...,38,2022-23,DEF,Spurs,0.0,0.00,0.00,0.0,0.0,0.0
120664,Ashley Young,0,NaN,NaN,NaN,0,0,0,NaN,NaN,...,38,2022-23,DEF,Aston Villa,1.0,0.00,0.00,0.0,0.0,0.0
120665,Jeremy Sarmiento Morante,0,NaN,NaN,NaN,0,0,0,NaN,NaN,...,38,2022-23,MID,Brighton,0.0,0.00,0.00,0.0,0.0,0.0


In [9]:
all_data.columns.tolist()

['name',
 'assists',
 'attempted_passes',
 'big_chances_created',
 'big_chances_missed',
 'bonus',
 'bps',
 'clean_sheets',
 'clearances_blocks_interceptions',
 'completed_passes',
 'creativity',
 'dribbles',
 'ea_index',
 'element',
 'errors_leading_to_goal',
 'errors_leading_to_goal_attempt',
 'fixture',
 'fouls',
 'goals_conceded',
 'goals_scored',
 'ict_index',
 'id',
 'influence',
 'key_passes',
 'kickoff_time',
 'kickoff_time_formatted',
 'loaned_in',
 'loaned_out',
 'minutes',
 'offside',
 'open_play_crosses',
 'opponent_team',
 'own_goals',
 'penalties_conceded',
 'penalties_missed',
 'penalties_saved',
 'recoveries',
 'red_cards',
 'round',
 'saves',
 'selected',
 'tackled',
 'tackles',
 'target_missed',
 'team_a_score',
 'team_h_score',
 'threat',
 'total_points',
 'transfers_balance',
 'transfers_in',
 'transfers_out',
 'value',
 'was_home',
 'winning_goals',
 'yellow_cards',
 'GW',
 'season',
 'position',
 'team',
 'xP',
 'expected_assists',
 'expected_goal_involvements',
 